# Chess evaluation — SE-ResNet (Kaggle `chessData.csv`)

**Google Colab (GPU T4).** Fast FEN → one-hot `(8,8,12)` + metadata via **Numba** (no `pandas.apply` / `eval` on FEN). **SE-ResNet** stem + residual blocks, **Flatten** + concat metadata, **Dense 512 → 256 → 1**. Labels: **WDL-style logistic** map to `[-1, 1]`.

**RAM:** ~1M float32 boards ≈ **3 GB**; use **high-RAM** runtime if needed.

**Runtime:** `Runtime` → `Change runtime type` → **GPU** (T4).

In [1]:
# @title Install dependencies
!pip install -q numba pandas tensorflow chess
# Optional: Kaggle download
# !pip install -q kaggle


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 85.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Configuration

Upload `chessData.csv` to `/content/` or set `CSV_PATH`. Match `COL_FEN` / `COL_EVAL` to your CSV headers.

In [2]:
from __future__ import annotations

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras import mixed_precision

from numba import njit, prange

# --- paths & training ---
CSV_PATH = "/content/chessData.csv"
N_ROWS = 500_000
COL_FEN = "FEN"
COL_EVAL = "Evaluation"

BATCH_SIZE = 4096  # try 8192 if VRAM allows
EPOCHS = 50
LEARNING_RATE = 3e-4
FILTERS = 128
NUM_SE_BLOCKS = 9
META_DIM = 8
IS_CHECK_WORKERS = 8  # 0 = skip python-chess in-check pass

EVAL_SCALE_CP = 400.0
CLIP_CP = 10_000.0
USE_MIXED_PRECISION = True

USE_KAGGLE_DOWNLOAD = False
KAGGLE_DATASET = "username/dataset-name"
KAGGLE_FILENAME = "chessData.csv"
CHECKPOINT_PATH = "/content/chess_se_resnet_best.keras"


In [3]:
def evaluations_to_centipawns(eval_series: pd.Series) -> np.ndarray:
    s = eval_series.astype(str).str.strip()
    out = pd.to_numeric(s, errors="coerce").to_numpy(dtype=np.float64)
    mate_m = s.str.extract(r"^#\s*([+-]?\d+)\s*$", expand=False)
    has_mate = mate_m.notna()
    if has_mate.any():
        mv = mate_m[has_mate].astype(np.int32).to_numpy()
        idx = has_mate.to_numpy()
        mvf = mv.astype(np.float64)
        out[idx] = np.sign(mvf) * (10_000.0 + np.abs(mvf))
    return out


def centipawns_to_target(cp: np.ndarray) -> np.ndarray:
    cp = np.clip(cp.astype(np.float64), -CLIP_CP, CLIP_CP)
    win_prob = 1.0 / (1.0 + np.power(10.0, -cp / EVAL_SCALE_CP))
    return (2.0 * win_prob - 1.0).astype(np.float32)


In [4]:
_PIECE_LUT = np.full(128, -1, dtype=np.int8)
for c, i in zip("PNBRQK", range(0, 6)):
    _PIECE_LUT[ord(c)] = i
for c, i in zip("pnbrqk", range(6, 12)):
    _PIECE_LUT[ord(c)] = i


@njit(parallel=True, cache=True)
def _encode_board_meta_batch(
    fen_bytes: np.ndarray,
    offsets: np.ndarray,
    lengths: np.ndarray,
    piece_lut: np.ndarray,
    out_board: np.ndarray,
    out_meta: np.ndarray,
):
    n = lengths.shape[0]
    for i in prange(n):
        start = offsets[i]
        L = lengths[i]
        sp = np.empty(5, dtype=np.int32)
        sp_count = 0
        for j in range(L):
            c = fen_bytes[start + j]
            if c == 32:
                sp[sp_count] = j
                sp_count += 1
                if sp_count == 5:
                    break
        if sp_count < 5:
            continue
        board_s0 = 0
        board_s1 = sp[0]
        turn_s0 = sp[0] + 1
        turn_s1 = sp[1]
        cast_s0 = sp[1] + 1
        cast_s1 = sp[2]
        ep_s0 = sp[2] + 1
        ep_s1 = sp[3]

        rank = 0
        file = 0
        for j in range(board_s0, board_s1):
            ch = fen_bytes[start + j]
            if ch == 47:
                rank += 1
                file = 0
                continue
            if rank >= 8:
                break
            if 49 <= ch <= 56:
                empties = ch - 48
                for _ in range(empties):
                    if file >= 8:
                        break
                    for p in range(12):
                        out_board[i, rank, file, p] = 0.0
                    file += 1
            else:
                if file < 8:
                    pidx = -1
                    if ch < 128:
                        pidx = piece_lut[ch]
                    for p in range(12):
                        out_board[i, rank, file, p] = 0.0
                    if 0 <= pidx < 12:
                        out_board[i, rank, file, pidx] = 1.0
                    file += 1

        wtm = 1.0
        if turn_s1 > turn_s0:
            tch = fen_bytes[start + turn_s0]
            if tch == 98:
                wtm = 0.0
        out_meta[i, 0] = wtm

        cwK = cwQ = cbK = cbQ = 0.0
        for j in range(cast_s0, cast_s1):
            ch = fen_bytes[start + j]
            if ch == 75:
                cwK = 1.0
            elif ch == 81:
                cwQ = 1.0
            elif ch == 107:
                cbK = 1.0
            elif ch == 113:
                cbQ = 1.0
        out_meta[i, 1] = cwK
        out_meta[i, 2] = cwQ
        out_meta[i, 3] = cbK
        out_meta[i, 4] = cbQ

        ep_pres = 0.0
        ep_file_norm = 0.0
        if ep_s1 > ep_s0:
            ech = fen_bytes[start + ep_s0]
            if ech != 45:
                ep_pres = 1.0
                fch = fen_bytes[start + ep_s0]
                if 97 <= fch <= 104:
                    ep_file_norm = (fch - 97) / 7.0
        out_meta[i, 5] = ep_pres
        out_meta[i, 6] = ep_file_norm
        out_meta[i, 7] = 0.0


def pack_fens_for_numba(fens: list[str]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = len(fens)
    lengths = np.empty(n, dtype=np.int32)
    for i, fen in enumerate(fens):
        lengths[i] = len(fen)
    total = int(lengths.sum())
    buf = np.empty(total, dtype=np.uint8)
    offsets = np.empty(n, dtype=np.int32)
    off = 0
    for i, fen in enumerate(fens):
        offsets[i] = off
        for j, ch in enumerate(fen):
            buf[off + j] = ord(ch)
        off += lengths[i]
    return buf, offsets, lengths


def fens_to_tensors_batch(fens: list[str]) -> tuple[np.ndarray, np.ndarray]:
    n = len(fens)
    boards = np.zeros((n, 8, 8, 12), dtype=np.float32)
    meta = np.zeros((n, META_DIM), dtype=np.float32)
    buf, offsets, lengths = pack_fens_for_numba(fens)
    _encode_board_meta_batch(buf, offsets, lengths, _PIECE_LUT, boards, meta)
    return boards, meta


In [5]:
def _is_check_one_fen(fen: str) -> float:
    """Top-level for multiprocessing pickle (nested functions fail on Pool)."""
    try:
        import chess
        return 1.0 if chess.Board(fen).is_check() else 0.0
    except Exception:
        return 0.0


def fill_in_check_metadata(fens: list[str], meta: np.ndarray, workers: int) -> None:
    if workers <= 0 or meta.shape[1] < 8:
        return
    try:
        import chess  # noqa: F401
    except ImportError:
        return

    from multiprocessing import Pool

    chunksize = max(1, len(fens) // (workers * 64))
    with Pool(workers) as pool:
        meta[:, 7] = np.asarray(
            pool.map(_is_check_one_fen, fens, chunksize=chunksize),
            dtype=np.float32,
        )


def make_dataset_from_arrays(boards, meta, y, shuffle: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices(((boards, meta), y))
    if shuffle:
        ds = ds.shuffle(min(100_000, len(y)), reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def preprocess_csv_to_numpy(csv_path: str, n_rows: int):
    print(f"Reading {n_rows:,} rows from {csv_path} ...")
    df = pd.read_csv(csv_path, nrows=n_rows, usecols=[COL_FEN, COL_EVAL])
    df = df.dropna(subset=[COL_FEN, COL_EVAL])
    fens = df[COL_FEN].tolist()
    cp = evaluations_to_centipawns(df[COL_EVAL])
    y = centipawns_to_target(cp)
    del df

    print(f"Parsing {len(fens):,} FENs with Numba ...")
    boards, meta = fens_to_tensors_batch(fens)
    if IS_CHECK_WORKERS > 0:
        print("Computing in-check metadata (python-chess) ...")
        fill_in_check_metadata(fens, meta, IS_CHECK_WORKERS)
    del fens
    return boards, meta, y


In [6]:
def se_block(x, channels: int, ratio: int = 4):
    s = layers.GlobalAveragePooling2D()(x)
    s = layers.Dense(max(channels // ratio, 8), activation="relu", use_bias=False)(s)
    s = layers.Dense(channels, activation="sigmoid", use_bias=False)(s)
    s = layers.Reshape((1, 1, channels))(s)
    return layers.Multiply()([x, s])


def res_se_block(x, filters: int):
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    y = se_block(y, filters)
    y = layers.Add()([x, y])
    return layers.Activation("relu")(y)


def build_model() -> Model:
    board_in = layers.Input(shape=(8, 8, 12), name="board")
    meta_in = layers.Input(shape=(META_DIM,), name="meta")

    x = layers.Conv2D(FILTERS, 3, padding="same", use_bias=False, name="stem")(board_in)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    for _ in range(NUM_SE_BLOCKS):
        x = res_se_block(x, FILTERS)

    x = layers.Flatten()(x)
    x = layers.Lambda(lambda t: tf.cast(t, tf.float32), name="board_flat_f32")(x)
    z = layers.Concatenate()([x, meta_in])
    z = layers.Dense(512, activation="relu", dtype="float32")(z)
    z = layers.Dense(256, activation="relu", dtype="float32")(z)
    out = layers.Dense(1, activation="linear", name="eval", dtype="float32")(z)

    return Model(inputs=[board_in, meta_in], outputs=out, name="chess_se_resnet_meta")


## Optional: download from Kaggle

Uncomment the pip line in the first cell, add `kaggle.json` under `/root/.kaggle/`, set `USE_KAGGLE_DOWNLOAD = True` and dataset slug in the config cell, then run this cell.

In [7]:
# Optional Kaggle download (edit USE_KAGGLE_DOWNLOAD + KAGGLE_* in config cell first)
if USE_KAGGLE_DOWNLOAD:
    os.makedirs("/root/.kaggle", exist_ok=True)
    assert os.path.exists("/root/.kaggle/kaggle.json"), "Add kaggle.json to /root/.kaggle/"
    os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p /content -f {KAGGLE_FILENAME}")
    os.system(f"unzip -o /content/{KAGGLE_FILENAME} -d /content")


## Train

Ensure `CSV_PATH` exists, then run the cell below.

In [8]:
!nvidia-smi

Sat Apr 25 20:00:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
if USE_MIXED_PRECISION:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: mixed_float16")

if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"Missing CSV: {CSV_PATH}")

boards, meta, y = preprocess_csv_to_numpy(CSV_PATH, N_ROWS)
n = len(y)
val_n = max(1, int(0.02 * n))
train_n = n - val_n

rng = np.random.default_rng(42)
perm = rng.permutation(n)
tr_idx, va_idx = perm[:train_n], perm[train_n:]

train_ds = make_dataset_from_arrays(boards[tr_idx], meta[tr_idx], y[tr_idx], shuffle=True)
val_ds = make_dataset_from_arrays(boards[va_idx], meta[va_idx], y[va_idx], shuffle=False)

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss=keras.losses.Huber(delta=1.0),
    metrics=["mae"],
)
callbacks = [
    keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, save_best_only=True, monitor="val_loss", mode="min"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

model.summary()
print(f"Train: {train_n:,} | Val: {val_n:,} | batch={BATCH_SIZE}")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
print("Done. Best checkpoint:", CHECKPOINT_PATH)


Mixed precision: mixed_float16
Reading 500,000 rows from /content/chessData.csv ...
Parsing 500,000 FENs with Numba ...
Computing in-check metadata (python-chess) ...


Model: "chess_se_resnet_meta"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ board (InputLayer)  │ (None, 8, 8, 12)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem (Conv2D)       │ (None, 8, 8, 128) │     13,824 │ board[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 8, 8, 128) │        512 │ stem[0][0]        │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 8, 8, 128) │    147,456 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 8, 8, 128) │    147,456 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      4,096 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      4,096 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 128) │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 8, 8, 128) │          0 │ activation[0][0], │
│                     │                   │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 8, 8, 128) │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 8, 8, 128) │    147,456 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 7,081,985 (27.02 MB)

 Trainable params: 7,077,121 (27.00 MB)

 Non-trainable params: 4,864 (19.00 KB)

Train: 490,000 | Val: 10,000 | batch=4096
Epoch 1/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 101s 476ms/step - loss: 0.6326 - mae: 0.9654 - val_loss: 0.1188 - val_mae: 0.3661 - learning_rate: 3.0000e-04
Epoch 2/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 33s 273ms/step - loss: 0.0978 - mae: 0.3398 - val_loss: 0.2301 - val_mae: 0.5833 - learning_rate: 3.0000e-04
Epoch 3/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 34s 280ms/step - loss: 0.0724 - mae: 0.2858 - val_loss: 0.2113 - val_mae: 0.5436 - learning_rate: 3.0000e-04
Epoch 4/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 34s 283ms/step - loss: 0.0570 - mae: 0.2541 - val_loss: 0.1425 - val_mae: 0.4167 - learning_rate: 1.5000e-04
Epoch 5/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 36s 299ms/step - loss: 0.0459 - mae: 0.2284 - val_loss: 0.0842 - val_mae: 0.3155 - learning_rate: 1.5000e-04
Epoch 6/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 36s 298ms/step - loss: 0.0373 - mae: 0.2068 - val_loss: 0.0779 - val_mae: 0.3119 - learning_rate: 1.5000e-04
Epoch 7/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 36s 299ms/step - loss: 

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
from google.colab import files

# تحميل الملف مباشرة لجهازك
print("Downloading the best model to your computer...")
files.download('/content/chess_se_resnet_best.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>